### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Step 2: Open the project directory

Ensure the dataset is in `emg2qwerty/data/` (with .hdf5 files). Adjust the path if your project is elsewhere.

In [2]:
%cd /content/drive/MyDrive/CS247A/emg2qwerty

/content/drive/MyDrive/CS247A/emg2qwerty


### Step 2a: Verify dataset

Place the dataset in `emg2qwerty/data/` (with .hdf5 session files inside) before running. The config expects `data/{session_name}.hdf5`.

In [9]:
import os

# Required sessions for single_user (subject 89335547)
REQUIRED_SESSIONS = [
    "2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-07-21-1626917264-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-05-1622889105-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-04-1622861066-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-07-22-1627001995-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-05-1622884635-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-07-21-1626915176-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",
    "2021-06-04-1622862148-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",  # val
    "2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f",  # test
]

data_dir = os.path.join(os.getcwd(), "data")
missing = [s for s in REQUIRED_SESSIONS if not os.path.exists(os.path.join(data_dir, f"{s}.hdf5"))]

if not missing:
    print(f"All 18 required .hdf5 files found in {data_dir}/")
else:
    print(f"MISSING {len(missing)} files in {data_dir}/")
    for s in missing[:3]:
        print(f"  - {s}.hdf5")
    if len(missing) > 3:
        print(f"  ... and {len(missing)-3} more")
    print("\nDownload from UCLA Box: https://ucla.app.box.com/s/3xc4nwpfjfpo6ydjs94t0v2kuq37d5eg")

All 18 required .hdf5 files found in /content/drive/MyDrive/CS247A/emg2qwerty/data/


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [4]:
%pip install -r requirements.txt

  Using cached https://github.com/kpu/kenlm/archive/master.zip (553 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


### Step 4: Start your experiments!

- The dataset is in `data/` (place .hdf5 files in `emg2qwerty/data/` before running).
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.
- **Model options:** Use `model=tds_conv_ctc` (baseline) or `model=transformer_ctc` (transformer).

In [32]:
# Option A: TDS Conv baseline (default)
# !python -m emg2qwerty.train \
#   user="single_user" \
#   model=tds_conv_ctc \
#   trainer.accelerator=gpu trainer.devices=1 \
#   # --multirun

# Option B: Transformer baseline
!python -m emg2qwerty.train \
  user="single_user" \
  model=transformer_ctc \
  trainer.accelerator=gpu trainer.devices=1 \
  num_workers=2 \
  trainer.max_epochs=10 \
  batch_size=64 \
  +trainer.precision=16 \
  # --multirun

[2026-02-25 05:22:25,895][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [ ]:
import os
import glob
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

# Find most recent run (PL: logs/date/time/lightning_logs/version_0/ or logs/date/time/)
log_dirs = sorted(glob.glob("logs/*/*/lightning_logs/version_*"), key=os.path.getmtime, reverse=True)
if not log_dirs:
    log_dirs = sorted(glob.glob("logs/*/*"), key=os.path.getmtime, reverse=True)
log_dir = log_dirs[0] if log_dirs else None

if log_dir is None:
    print("No logs found. Run training first.")
else:
    print(f"Loading: {log_dir}")
    ea = EventAccumulator(log_dir)
    ea.Reload()
    tags = ea.Tags().get("scalars", [])

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # Y-axis limits for first 2 plots (set to None for auto)
    loss_ylim = (0, 5)   # e.g. (0, 5) or None
    cer_ylim = (0, 100)  # e.g. (0, 100) or None

    # 1. Loss
    for tag in ["train/loss", "val/loss"]:
        if tag in tags:
            events = ea.Scalars(tag)
            steps, vals = [e.step for e in events], [e.value for e in events]
            axes[0, 0].plot(steps, vals, label=tag, marker=".", markersize=3)
    axes[0, 0].set_xlabel("Step")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].set_title("Training & Validation Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    if loss_ylim is not None:
        axes[0, 0].set_ylim(loss_ylim)

    # 2. CER
    for tag in ["train/CER", "val/CER"]:
        if tag in tags:
            events = ea.Scalars(tag)
            steps, cer = [e.step for e in events], [e.value for e in events]
            axes[0, 1].plot(steps, cer, label=tag, marker=".", markersize=3)
    axes[0, 1].set_xlabel("Step")
    axes[0, 1].set_ylabel("CER (%)")
    axes[0, 1].set_title("Character Error Rate (lower = better)")
    axes[0, 1].axhline(y=100, color="gray", linestyle="--", alpha=0.5)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    if cer_ylim is not None:
        axes[0, 1].set_ylim(cer_ylim)

    # 3. Error breakdown (IER, DER, SER) - val only
    for tag, color in [("val/IER", "C0"), ("val/DER", "C1"), ("val/SER", "C2")]:
        if tag in tags:
            events = ea.Scalars(tag)
            steps, vals = [e.step for e in events], [e.value for e in events]
            axes[1, 0].plot(steps, vals, label=tag.replace("val/", ""), color=color, marker=".", markersize=3)
    axes[1, 0].set_xlabel("Step")
    axes[1, 0].set_ylabel("Rate (%)")
    axes[1, 0].set_title("Error Breakdown (Insertion, Deletion, Substitution)")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Learning rate (if logged by LearningRateMonitor)
    lr_tags = [t for t in tags if "lr" in t.lower() or "learning" in t.lower()]
    if lr_tags:
        for tag in lr_tags[:2]:  # max 2 LR curves
            events = ea.Scalars(tag)
            steps, lrs = [e.step for e in events], [e.value for e in events]
            axes[1, 1].plot(steps, lrs, label=tag, marker=".", markersize=3)
        axes[1, 1].set_xlabel("Step")
        axes[1, 1].set_ylabel("Learning Rate")
        axes[1, 1].set_title("Learning Rate Schedule")
    else:
        axes[1, 1].text(0.5, 0.5, "No LR logs", ha="center", va="center", transform=axes[1, 1].transAxes)
        axes[1, 1].set_title("Learning Rate (not logged)")
    if lr_tags:
        axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

#### Testing

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.
- Use `model=transformer_ctc` when evaluating a transformer checkpoint; use `model=tds_conv_ctc` (or omit) for the TDS baseline.

In [34]:
# Testing (use model=transformer_ctc for transformer checkpoints)
!python -m emg2qwerty.train \
  user="single_user" \
  model=transformer_ctc \
  checkpoint='/content/drive/MyDrive/CS247A/emg2qwerty/logs/2026-02-25/04-54-16/checkpoints/epoch\=1-step\=240.ckpt' \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy \
  hydra.launcher.mem_gb=64 \
  # --multirun

[2026-02-25 05:41:10,986][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f